# 10 - 代理式 RAG（自主代理）

**复杂度：** ⭐⭐⭐⭐⭐

**应用场景：** 多步推理、商业智能仪表板、复杂决策制定

**核心特性：**
- ReAct 代理（推理 + 行动）
- 多种工具（检索器、计算器、网络搜索）
- 自主规划和执行
- 多步推理能力

**示例：**
```
Query: "If I have 10K docs and process 1M tokens/day, should I use OpenAI or HF?"

Agent:
1. Thought: Need cost calculation
   Action: Calculator → Estimate costs
2. Thought: Need comparison info
   Action: Knowledge Base → Retrieve comparison
3. Thought: Can now recommend
   Final Answer: "HuggingFace is better because..."
```

In [17]:
import sys
sys.path.append('../..')

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL, LLM_PROVIDER
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import REACT_AGENT_PROMPT

print_section_header("设置：代理式 RAG")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

print("✅ 设置完成！")


设置：代理式 RAG



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings
✅ 设置完成！


## 2. 为代理创建工具

In [18]:
from langchain_core.tools import tool
import numexpr

print_section_header("代理工具")

# 工具 1：知识库检索器
@tool
def knowledge_base(query: str) -> str:
    """Search LangChain RAG knowledge base.
    Use for questions about RAG, embeddings, retrievers, LangChain."""
    docs = retriever.invoke(query)
    return format_docs(docs[:3])

# 工具 2：计算器
@tool
def calculator(expression: str) -> str:
    """Calculate mathematical expressions safely.
    Input should be a valid math expression like '2 + 2' or '100000 * 0.00002'.
    Examples: '10 + 5', '(100 * 2) / 3', '2 ** 8'"""
    try:
        # 使用 numexpr 进行安全求值
        result = numexpr.evaluate(expression).item()
        return str(result)
    except Exception as e:
        return f"Error calculating '{expression}': {str(e)}"

# 工具 3：网络搜索（如果可用）
tools = [knowledge_base, calculator]

try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    
    @tool
    def web_search(query: str) -> str:
        """Search the web for current information.
        Use for queries outside the knowledge base."""
        search = TavilySearchResults(max_results=2)
        return search.invoke(query)
    
    tools.append(web_search)
    print("✓ 创建了 3 个工具（检索器、计算器、网络搜索）")
except ImportError:
    print("✓ 创建了 2 个工具（检索器、计算器）")

print("  工具列表：")
for t in tools:
    print(f"    - {t.name}: {t.description[:60]}...")


代理工具

✓ 创建了 3 个工具（检索器、计算器、网络搜索）
  工具列表：
    - knowledge_base: Search LangChain RAG knowledge base.
    Use for questions a...
    - calculator: Calculate mathematical expressions safely.
    Input should ...
    - web_search: Search the web for current information.
        Use for quer...


## 3. 创建 ReAct 代理

In [19]:
from langchain.agents import create_agent

print_section_header("现代 ReAct 代理（LangChain 1.0）")

# DeepSeek 思考模式会返回非标准的 `reasoning_content` 字段。
# LangChain 的通用 ChatOpenAI 代理循环不会在工具调用之间保留它，
# 因此我们为此笔记本的工具调用代理禁用思考功能。
agent_llm = llm
if LLM_PROVIDER == "deepseek":
    agent_llm = create_chat_model(
        model=DEFAULT_MODEL,
        temperature=0,
        extra_body={"thinking": {"type": "disabled"}},
    )

# 创建现代代理（基于 LangGraph 构建）
agent_executor = create_agent(
    model=agent_llm,
    tools=tools,
    system_prompt="""你是一个在 RAG 系统和嵌入方面具有专业知识的有用 AI 助手。

你可以使用以下工具：
- knowledge_base：搜索有关 RAG、嵌入和 LangChain 的文档
- calculator：执行数学计算
- web_search：搜索网络获取最新信息（如果可用）

指导原则：
1. 逐步思考你需要什么信息
2. 使用适当的工具收集信息
3. 将结果综合成清晰、简洁的答案
4. 使用知识库时始终引用来源

保持乐于助人和准确！"""
)

print("✓ 现代 ReAct 代理已创建")
print("  - 内部基于 LangGraph 构建")
print("  - 自动状态管理")
print("  - 基于消息的接口")
print("  - 流式支持")
if LLM_PROVIDER == "deepseek":
    print("  - 禁用 DeepSeek 思考模式以保持工具调用轮次兼容")


现代 REACT 代理（LANGCHAIN 1.0）

✓ 现代 ReAct 代理已创建
  - 内部基于 LangGraph 构建
  - 自动状态管理
  - 基于消息的接口
  - 流式支持
  - 禁用 DeepSeek 思考模式以保持工具调用轮次兼容


## 4. 测试代理 - 简单查询

In [20]:
print_section_header("测试 1：简单查询")

query = "What is a vector store?"
print(f"Query: '{query}'\n")
print("=" * 80)

# 现代基于消息的调用
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

print("\n" + "=" * 80)

# 从消息中提取最终答案
final_answer = response["messages"][-1].content
print(f"\n最终答案：{final_answer}")


测试 1：简单查询

Query: 'What is a vector store?'



最终答案：Based on the knowledge base, here's a clear explanation:

## What is a Vector Store?

A **vector store** (also called a vector database) is a specialized storage system designed to store and query **embeddings** — numerical vector representations of data (like text, images, or audio).

### How it works in RAG:

1. **Embedding**: Documents are split into chunks, and each chunk is converted into a vector (a list of numbers) using an embedding model. These vectors capture the **semantic meaning** of the text.

2. **Storing**: The embeddings are inserted into the vector store alongside the original text content.

3. **Querying**: When a user asks a question, the question is also converted into an embedding. The vector store then performs a **similarity search** (e.g., cosine similarity) to find the most semantically relevant document chunks.

### Key characteristics:

- **Semantic search**: Unlike traditional keyword-based search, vector

## 5. 测试代理 - 多步推理

In [21]:
print_section_header("测试 2：多步推理")

query = """If embedding 100,000 documents costs $2 with OpenAI,
what's the cost per document? Then tell me if HuggingFace is cheaper."""

print(f"Query: {query}\n")
print("=" * 80)

# 现代基于消息的调用
response = agent_executor.invoke({
    "messages": [{"role": "user", "content": query}]
})

print("\n" + "=" * 80)

# 提取最终答案
final_answer = response["messages"][-1].content
print(f"\n最终答案：{final_answer}")

print("\n💡 代理使用了多个工具来解决这个问题！")


测试 2：多步推理

Query: If embedding 100,000 documents costs $2 with OpenAI,
what's the cost per document? Then tell me if HuggingFace is cheaper.



最终答案：The web search tool is currently unavailable. Let me answer based on what I know.

Here's the answer:

---

## Cost per Document

**$2 ÷ 100,000 documents = $0.00002 per document** (that's **2/100ths of a cent** per document).

## Is HuggingFace Cheaper?

**Yes, HuggingFace is typically much cheaper — often free.**

Here's why:

1. **OpenAI** (e.g., `text-embedding-ada-002` or `text-embedding-3-small`): You pay per token used. At roughly $0.02–$0.13 per 1M tokens depending on the model, the cost scales with usage.

2. **HuggingFace models** (e.g., `all-MiniLM-L6-v2`, `BAAI/bge-small-en-v1.5`): These are **open-source models you can run locally on your own hardware** (CPU or GPU). There are **no per-document or per-token API fees** — you only pay for the compute infrastructure (e.g., a server or your own machine).

### Cost Comparison Summ

## 6. 带记忆的代理

In [22]:
print_section_header("对话式代理")

print("✓ 使用基于消息的对话状态\n")
print("  现代 LangChain 1.0 方法：")
print("  - 不需要单独的内存对象")
print("  - 通过消息历史管理状态")
print("  - 代理自动维护上下文\n")

# 将对话历史维护为消息列表
conversation = []

# 多轮对话
# 第 1 轮
query1 = "What are the embedding dimensions?"
print(f"User: {query1}\n")

conversation.append({"role": "user", "content": query1})
response1 = agent_executor.invoke({"messages": conversation})
conversation = response1["messages"]  # 用完整对话更新

answer1 = conversation[-1].content
print(f"Agent: {answer1[:150]}...\n")

# 第 2 轮（代理有第 1 轮的上下文）
query2 = "Which one is cheaper?"
print(f"\nUser: {query2}\n")

conversation.append({"role": "user", "content": query2})
response2 = agent_executor.invoke({"messages": conversation})
conversation = response2["messages"]  # 用完整对话更新

answer2 = conversation[-1].content
print(f"Agent: {answer2[:150]}...")

print("\n✅ 代理保持了对话上下文！")
print(f"   （对话中的总消息数：{len(conversation)}）")


对话式代理

✓ 使用基于消息的对话状态

  现代 LangChain 1.0 方法：
  - 不需要单独的内存对象
  - 通过消息历史管理状态
  - 代理自动维护上下文

User: What are the embedding dimensions?

Agent: The knowledge base doesn't contain a dedicated reference table for embedding dimensions. Let me provide you with a comprehensive answer based on widel...


User: Which one is cheaper?

Agent: Unfortunately, the web search tool is currently unavailable. However, I can share the well-known pricing information based on my training data. Here's...

✅ 代理保持了对话上下文！
   （对话中的总消息数：30）


## 总结

**流程：**
```
查询 → 代理推理循环：
  1. 思考要做什么
  2. 选择要使用的工具
  3. 执行工具
  4. 观察结果
  5. 重复直到找到答案
→ 最终答案
```

**优势：**
✅ 自主决策  
✅ 多步推理  
✅ 工具组合  
✅ 处理复杂查询  
✅ 可扩展（添加更多工具）  

**局限性：**
- 非常慢（多次 LLM 调用）
- 昂贵（推理 tokens）
- 可能陷入循环
- 行为不可预测
- 难以调试

**何时使用：**
- 复杂的多步任务
- 需要多个数据源
- 研究/分析工作流
- 商业智能仪表板

**生产环境建议：**
- 谨慎设置 max_iterations
- 监控 token 使用量
- 添加超时机制
- 实现回退方案
- 使用 LangSmith 进行追踪
- 彻底测试边缘情况

**下一步：** [11_comparison.ipynb](11_comparison.ipynb) - 基准测试所有架构